In [ ]:
# Install required libraries
!pip install -qU langchain langchain-core langchain-google-genai langchain-tavily langchain-text-splitters langchain-huggingface sentence-transformers pymilvus

In [ ]:
import os
import re
import json
import time
import hashlib
import sqlite3

from google.colab import userdata

from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch, TavilyExtract
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from pymilvus import MilvusClient, DataType

os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

MILVUS_URI = userdata.get("MILVUS_URI")
MILVUS_TOKEN = userdata.get("MILVUS_TOKEN")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

search_tool = TavilySearch(
    max_results=5,
    search_depth="advanced"
)

extract_tool = TavilyExtract(
    extract_depth="advanced",
    format="markdown"
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


client = MilvusClient(
    uri=MILVUS_URI,
    token=MILVUS_TOKEN
)

collection_name = "company_documents"

companies = {
    "microsoft": "Microsoft",
    "google": "Google",
    "amazon": "Amazon"
}

print("All services initialized successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

All services initialized successfully!


In [ ]:
search_results = []


@tool
def search_company_web(company: str, topic: str):
    """
    Search the web for company documents related to a topic.
    Topics can include financial reports, blogs and press releases.
    """

    query = f"{company} {topic}"

    results = search_tool.invoke({
        "query": query
    })

    if isinstance(results, str):
        results = json.loads(results)

    for item in results.get("results", []):

        search_results.append({
            "company": company.lower(),
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "content": item.get("content", ""),
            "category": topic
        })

    return json.dumps(results)


research_agent = create_agent(
    model=llm,
    tools=[search_company_web],
    system_prompt="""
    You are a company research agent.

    Your task is to find useful documents about a company.

    You must use the search_company_web tool for:
    1. Financial reports and revenue.
    2. Company blogs and business developments.
    3. Official press releases.

    Search for all three topics.
    Do not invent search results.
    """
)

for company_key, company_name in companies.items():

    print("Researching:", company_name)

    max_retries = 5
    delay = 2
    for attempt in range(max_retries):
        try:
            research_agent.invoke({
                "messages": [
                    {
                        "role": "user",
                        "content": f"""
                        Find financial reports, blogs and press releases
                        related to {company_name}.
                        Use the search tool for all three topics.
                        """
                    }
                ]
            })
            break
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            print(f"  Attempt {attempt + 1} failed with error: {e}. Retrying in {delay} seconds...")
            time.sleep(delay)
            delay *= 2



unique_results = {}

for item in search_results:

    if item["url"]:
        unique_results[item["url"]] = item

search_results = list(unique_results.values())


urls = [
    item["url"]
    for item in search_results
]

extracted_documents = []

if urls:

    extraction = extract_tool.invoke({
        "urls": urls[:20]
    })

    for result in extraction.get("results", []):

        matching = next(
            (item for item in search_results
             if item["url"] == result["url"]),
            {}
        )

        content = result.get("raw_content", "")

        if content.strip():

            extracted_documents.append({
                "company": matching["company"],
                "title": matching["title"],
                "source_url": result["url"],
                "category": matching["category"],
                "content": content
            })


chat_history = [
    {
        "company": "microsoft",
        "content": "We discussed Microsoft's revenue, Azure growth and financial performance."
    },
    {
        "company": "google",
        "content": "We discussed Google's advertising business, AI investments and cloud services."
    },
    {
        "company": "amazon",
        "content": "We discussed Amazon's e-commerce business, AWS and operating income."
    }
]


for chat in chat_history:

    extracted_documents.append({
        "company": chat["company"],
        "title": "Previous Chat History",
        "source_url": "chat_history",
        "category": "chat_history",
        "content": chat["content"]
    })


print("\nResearch completed!")
print("Search results:", len(search_results))
print("Documents collected:", len(extracted_documents))

Researching: Microsoft
Researching: Google
Researching: Amazon

Research completed!
Search results: 45
Documents collected: 21


In [ ]:

documents = []

for item in extracted_documents:

    text = re.sub(r"\s+", " ", item["content"]).strip()

    if not text:
        continue

    documents.append(
        Document(
            page_content=text,
            metadata={
                "company": item["company"],
                "title": item["title"][:512],
                "source_url": item["source_url"][:2048],
                "category": item["category"]
            }
        )
    )

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunked_documents = text_splitter.split_documents(documents)

print("Total chunks:", len(chunked_documents))


schema = client.create_schema(
    auto_id=False,
    enable_dynamic_field=False
)

schema.add_field(
    field_name="id",
    datatype=DataType.INT64,
    is_primary=True
)

schema.add_field(
    field_name="vector",
    datatype=DataType.FLOAT_VECTOR,
    dim=384
)

schema.add_field(
    field_name="text",
    datatype=DataType.VARCHAR,
    max_length=65535
)

schema.add_field(
    field_name="company",
    datatype=DataType.VARCHAR,
    max_length=128
)

schema.add_field(
    field_name="title",
    datatype=DataType.VARCHAR,
    max_length=512
)

schema.add_field(
    field_name="source_url",
    datatype=DataType.VARCHAR,
    max_length=2048
)

schema.add_field(
    field_name="category",
    datatype=DataType.VARCHAR,
    max_length=128
)


index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)


if client.has_collection(collection_name):
    print(f"Collection '{collection_name}' already exists. Dropping it to recreate with new ingestion.")
    client.drop_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params
)

for company in companies.keys():

    client.create_partition(
        collection_name=collection_name,
        partition_name=company
    )


for index, document in enumerate(chunked_documents):

    company = document.metadata["company"]

    vector = embeddings.embed_query(
        document.page_content
    )

    unique_text = (
        company
        + document.metadata["source_url"]
        + str(index)
    )

    doc_id = int(
        hashlib.sha256(
            unique_text.encode()
        ).hexdigest()[:15],
        16
    )

    record = {
        "id": doc_id,
        "vector": vector,
        "text": document.page_content,
        "company": company,
        "title": document.metadata["title"],
        "source_url": document.metadata["source_url"],
        "category": document.metadata["category"]
    }

    client.insert(
        collection_name=collection_name,
        data=[record],
        partition_name=company
    )



client.load_collection(collection_name)

print("\nIngestion completed!")
print("Company partitions:", client.list_partitions(collection_name))

Total chunks: 1417
Collection 'company_documents' already exists. Dropping it to recreate with new ingestion.

Ingestion completed!
Company partitions: ['_default', 'microsoft', 'google', 'amazon']


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create query construction prompt
query_prompt = ChatPromptTemplate.from_template("""
You are a query construction assistant for a business RAG system.

Company: {company}

Previous chat history:
{history}

User question:
{question}

Instructions:
1. Understand the question using chat history.
2. Rewrite it as a clear standalone search query.
3. Generate 3 relevant subqueries.
4. Keep all queries related to the selected company.

Return valid JSON only:

{{
    "rewritten_query": "improved query",
    "sub_queries": [
        "query 1",
        "query 2",
        "query 3"
    ]
}}
""")



query_chain = query_prompt | llm | StrOutputParser()



def construct_queries(question, company, history):

    response = query_chain.invoke({
        "company": company,
        "history": history,
        "question": question
    })


    response = response.strip()

    if response.startswith("```"):
        response = response.split("```")[1]
        response = response.replace("json", "", 1)

    return json.loads(response.strip())



def search_documents(query, company, top_k=5):

    query_vector = embeddings.embed_query(query)

    results = client.search(
        collection_name=collection_name,
        data=[query_vector],
        anns_field="vector",
        limit=top_k,
        partition_names=[company],
        output_fields=[
            "text",
            "company",
            "title",
            "source_url",
            "category"
        ]
    )

    return results[0]


def retrieve_documents(constructed, company):

    queries = [
        constructed["rewritten_query"]
    ] + constructed["sub_queries"]

    retrieved = {}

    for query in queries:

        results = search_documents(
            query,
            company,
            top_k=5
        )

        for result in results:

            entity = result["entity"]

            retrieved[result["id"]] = {
                "text": entity["text"],
                "company": entity["company"],
                "title": entity["title"],
                "source_url": entity["source_url"],
                "category": entity["category"],
                "score": result["distance"]
            }

    return sorted(
        retrieved.values(),
        key=lambda item: item["score"],
        reverse=True
    )


question = "Compare its financial performance with last year."

company = "microsoft"

history = """
We previously discussed Microsoft's revenue,
Azure growth and financial performance.
"""

constructed = construct_queries(
    question,
    company,
    history
)

print("Rewritten query:")
print(constructed["rewritten_query"])

print("\nSubqueries:")
for query in constructed["sub_queries"]:
    print("-", query)


retrieved_documents = retrieve_documents(
    constructed,
    company
)

print("\nRetrieved documents:")

for document in retrieved_documents:

    print("\nTitle:", document["title"])
    print("Company:", document["company"])
    print("Score:", document["score"])
    print("Content:", document["text"][:250])
    print("Source:", document["source_url"])

Rewritten query:
Microsoft annual financial performance year-over-year comparison with previous fiscal year

Subqueries:
- Microsoft total revenue and net income year-over-year growth comparison
- Microsoft Azure and Intelligent Cloud segment revenue growth compared to last year
- Microsoft operating income, operating margins, and EPS comparison to previous year

Retrieved documents:

Title: Microsoft 2025 Annual Report
Company: microsoft
Score: 0.7622126936912537
Content: content (including games and in-game content), Xbox Game Pass and other subscriptions, Xbox Cloud Gaming, advertising, and other cloud services Search and news advertising revenue (ex TAC) growth Revenue from search and news advertising excluding tra
Source: https://www.microsoft.com/investor/reports/ar25/index.html

Title: Microsoft 2025 Annual Report
Company: microsoft
Score: 0.7529904246330261
Content: cloud. * Operating expenses increased $1.1 billion or 4% driven by investments in cloud and AI engineering and co

In [ ]:
# Create SQLite cache database
connection = sqlite3.connect("rag_cache.db")

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS cache (
    cache_key TEXT PRIMARY KEY,
    cache_value TEXT,
    created_at REAL
)
""")

connection.commit()


def create_cache_key(company, question, history, cache_type):

    value = (
        cache_type
        + company
        + question
        + history
    )

    return hashlib.sha256(
        value.encode()
    ).hexdigest()


def get_cache(key):

    cursor.execute(
        "SELECT cache_value, created_at FROM cache WHERE cache_key = ?",
        (key,)
    )

    result = cursor.fetchone()

    if result:

        cache_value, created_at = result


        if time.time() - created_at < 86400:

            print("Cache HIT")

            return json.loads(cache_value)


        cursor.execute(
            "DELETE FROM cache WHERE cache_key = ?",
            (key,)
        )

        connection.commit()

    print("Cache MISS")

    return None


def save_cache(key, value):

    cursor.execute(
        """
        INSERT OR REPLACE INTO cache
        (cache_key, cache_value, created_at)
        VALUES (?, ?, ?)
        """,
        (
            key,
            json.dumps(value),
            time.time()
        )
    )

    connection.commit()


def cached_retrieval(question, company, history):

    key = create_cache_key(
        company,
        question,
        history,
        "retrieval"
    )

    cached_result = get_cache(key)

    if cached_result is not None:
        return cached_result

    constructed = construct_queries(
        question,
        company,
        history
    )


    results = retrieve_documents(
        constructed,
        company
    )


    save_cache(key, results)

    return results


print("SQLite caching initialized successfully!")

SQLite caching initialized successfully!


In [ ]:
#Create answer generation prompt
answer_prompt = ChatPromptTemplate.from_template("""
You are a business analysis assistant.

Company: {company}

Previous conversation:
{history}

User question:
{question}

Retrieved documents:
{context}

Instructions:
1. Answer clearly and professionally.
2. Use the retrieved documents as evidence.
3. Do not invent financial figures.
4. Explain relevant business findings.
5. Include source URLs.
6. Mention if information is insufficient.

Provide a structured answer.
""")



answer_chain = answer_prompt | llm | StrOutputParser()


def generate_answer(question, company, history=""):

    key = create_cache_key(
        company,
        question,
        history,
        "answer"
    )


    cached_answer = get_cache(key)

    if cached_answer is not None:
        return cached_answer

    documents = cached_retrieval(
        question,
        company,
        history
    )

    if not documents:
        return "No relevant documents found."

    context = ""

    for document in documents[:5]:

        context += f"""
        Company: {document['company']}
        Title: {document['title']}
        Source: {document['source_url']}
        Content: {document['text']}
        """

    answer = answer_chain.invoke({
        "company": company,
        "history": history,
        "question": question,
        "context": context
    })


    save_cache(key, answer)

    return answer


question = "Explain the company's financial performance."

answer = generate_answer(
    question,
    "microsoft",
    "We previously discussed Microsoft's revenue and Azure growth."
)

print("MICROSOFT BUSINESS ANALYSIS")
print("=" * 50)
print(answer)


for company in companies.keys():

    print("\n")
    print("=" * 50)
    print("COMPANY:", company.upper())
    print("=" * 50)

    answer = generate_answer(
        question,
        company,
        ""
    )

    print(answer)


def compare_companies(question, company_list):

    company_context = ""

    for company in company_list:

        documents = cached_retrieval(
            question,
            company,
            ""
        )

        company_context += f"\nCompany: {company}\n"

        for document in documents[:5]:

            company_context += f"""
            Title: {document['title']}
            Source: {document['source_url']}
            Content: {document['text']}
            """

    comparison_prompt = ChatPromptTemplate.from_template("""
    You are a business analyst.

    Question:
    {question}

    Company information:
    {context}

    Compare the companies based only on the
    retrieved information.

    Explain similarities, differences and key findings.
    Include source URLs.
    Do not invent information.
    """)

    comparison_chain = (
        comparison_prompt
        | llm
        | StrOutputParser()
    )

    return comparison_chain.invoke({
        "question": question,
        "context": company_context
    })


comparison = compare_companies(
    "Compare the financial performance of Microsoft, Google and Amazon.",
    ["microsoft", "google", "amazon"]
)

print("\nMULTI-COMPANY COMPARISON")
print("=" * 50)
print(comparison)


print("\nTesting answer cache:")

answer = generate_answer(
    "Explain the company's financial performance.",
    "microsoft",
    "We previously discussed Microsoft's revenue and Azure growth."
)

print(answer)

Cache MISS
Cache MISS
MICROSOFT BUSINESS ANALYSIS
Based on the provided annual reports, here is an analysis of Microsoft's financial performance:

---

### 1. Overall Financial Performance (Fiscal Year 2025 vs. Fiscal Year 2024)

Microsoft demonstrated double-digit growth across all primary financial metrics in FY 2025:

* **Revenue:** **$281,724 million**, an increase of **15%** (+$36.6 billion) compared to $245,122 million in FY 2024. Growth was driven across all operating segments.
* **Gross Margin:** **$193,893 million**, up **13%** (+$22.9 billion) compared to $171,008 million in FY 2024.
* **Operating Income:** **$128,528 million**, an increase of **17%** (+$19.1 billion) compared to $109,433 million in FY 2024.
* **Net Income:** **$101,832 million**, up **16%** compared to $88,136 million in FY 2024.
* **Diluted Earnings Per Share (EPS):** **$13.64**, representing **16%** growth compared to $11.80 in FY 2024.

---

### 2. Segment Highlights and Revenue Drivers

* **Intelligent C

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 2h52m40.073034693s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': '